# Installing NRPy and Running a First Standalone Project

This notebook verifies an NRPy install, generates the standalone Cartesian wave-equation project, builds it, runs it, and inspects the first diagnostic file. No numerical relativity background is assumed.

[Index](../index.ipynb) | Previous: [Index](../index.ipynb) | Next: [NRPy Package Map](package_map.ipynb)

## Why This Matters

A first successful generated project checks the complete local workflow: Python can import NRPy, NRPy can write C code, `make` can build that code, and the executable can write diagnostics.

## What You Will See

- An import confirmation for the `nrpy` package.
- A generated Cartesian wave-equation project and file inventory.
- A `make` build and executable run.
- Diagnostic rows from the generated code.

## Table of Contents

1. Step 1: Prerequisites and install paths
2. Step 2: Verify the NRPy import
3. Step 3: Generate the Cartesian wave-equation project
4. Step 4: Inspect the generated tree
5. Step 5: Build the generated project
6. Step 6: Run the executable and inspect diagnostics
7. Step 7: Interpret the first success criterion
8. Step 8: Where this fits in the tutorial

## Step 1: Prerequisites and Install Paths

Learners install the latest package:

```bash
python -m pip install nrpy
```

Contributors working from an NRPy source checkout should follow that repository's current contributor setup instructions before returning here.

Installation commands are intentionally not run automatically. The import check below verifies that Python can load NRPy. Generation requires Python; build and run steps require `make` and a C compiler.

## Step 2: Verify the NRPy Import

This cell verifies that the notebook kernel can import NRPy. A **package** is installable Python code, and `import` is the command that loads that package into the running kernel.

In [1]:
import nrpy

print("nrpy import succeeded")

nrpy import succeeded


## Helper Functions and Workspace Paths

The remaining cells use a fresh temporary generator workspace. The NRPy generator recreates `project/wave_equation_cartesian/` relative to its working directory, so a temporary workspace prevents accidental deletion of an existing project.

In [2]:
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import tempfile

PROJECT_NAME = "wave_equation_cartesian"
_run_workspace = tempfile.TemporaryDirectory(prefix="nrpy-first-run-")
GENERATOR_CWD = Path(_run_workspace.name).resolve()
PROJECT_DIR = GENERATOR_CWD / "project" / PROJECT_NAME

print("Generator workspace: temporary directory")
print(f"Project directory: project/{PROJECT_NAME}")


def display_path(path):
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        if path == PROJECT_DIR:
            return f"project/{PROJECT_NAME}"
        if path == GENERATOR_CWD:
            return "temporary workspace"
        return path.name


def display_command(command):
    return " ".join("python" if str(part) == sys.executable else str(part) for part in command)


def clean_process_output(text):
    text = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text)
    return text.replace("\r", "\n")


def run_command(command, cwd, max_lines=40):
    cwd = Path(cwd)
    if not cwd.is_dir():
        raise FileNotFoundError(f"Working directory does not exist: {display_path(cwd)}")

    print(f"$ {display_command(command)}")
    print(f"cwd: {display_path(cwd)}")
    try:
        completed = subprocess.run(
            command,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
        )
    except subprocess.CalledProcessError as err:
        output = clean_process_output(err.stdout or "")
        lines = output.splitlines()
        if lines:
            print(f"Last {min(max_lines, len(lines))} output line(s):")
            print("\n".join(lines[-max_lines:]))
        raise

    output = clean_process_output(completed.stdout or "")
    lines = output.splitlines()
    if lines:
        print(f"Last {min(max_lines, len(lines))} output line(s):")
        print("\n".join(lines[-max_lines:]))
    else:
        print("(no output)")
    return completed


def require_tool(name):
    resolved = shutil.which(name)
    if resolved is None:
        raise RuntimeError(f"Required tool not found on PATH: {name}")
    print(f"{name}: available")
    return resolved


def require_c_compiler():
    cc_env = os.environ.get("CC")
    if cc_env:
        cc_path = shutil.which(cc_env)
        if cc_path is None:
            raise RuntimeError(f"CC is set to {cc_env!r}, but it does not resolve on PATH.")
        print(f"CC: {cc_env}")
        return cc_path

    found = []
    for compiler in ("cc", "gcc", "clang"):
        resolved = shutil.which(compiler)
        print(f"{compiler}: {'available' if resolved else 'not found'}")
        if resolved:
            found.append(resolved)
    if not found:
        raise RuntimeError("Required C compiler not found on PATH; install cc, gcc, clang, or set CC.")
    return found[0]


def first_lines(path, n=8):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"First {n} line(s) of {display_path(path)}:")
    with path.open("r", encoding="utf-8", errors="replace") as stream:
        for _, line in zip(range(n), stream):
            print(line.rstrip())


def ensure_safe_to_regenerate(project_dir):
    project_dir = Path(project_dir)
    if project_dir.exists() or project_dir.is_symlink():
        raise RuntimeError(
            "Generated project directory already exists. Choose a clean workspace "
            "before rerunning generation."
        )

Generator workspace: temporary directory
Project directory: project/wave_equation_cartesian


## Step 3: Generate the Cartesian Wave-Equation Project

The generator writes `project/wave_equation_cartesian/` relative to `GENERATOR_CWD`. It recreates that directory, so this notebook chooses a fresh temporary working directory before running generation.

In [3]:
ensure_safe_to_regenerate(PROJECT_DIR)
run_command(
    [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"],
    cwd=GENERATOR_CWD,
)
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)
print(f"Generated project: project/{PROJECT_NAME}")

$ python -m nrpy.examples.wave_equation_cartesian
cwd: temporary workspace


Last 2 output line(s):
Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par
Generated project: project/wave_equation_cartesian


## Step 4: Inspect the Generated Tree

A standalone BHaH project includes C sources, headers, a `Makefile`, a parameter file, helper directories, and later an executable and diagnostics.

In [4]:
required_paths = [
    PROJECT_DIR / "Makefile",
    PROJECT_DIR / f"{PROJECT_NAME}.par",
]
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing generated file(s): {missing}")

c_sources = sorted(PROJECT_DIR.rglob("*.c"))
headers = sorted(PROJECT_DIR.rglob("*.h"))
if not c_sources:
    raise FileNotFoundError(f"No generated C sources found under {PROJECT_DIR}")
if not headers:
    raise FileNotFoundError(f"No generated headers found under {PROJECT_DIR}")

print("Required files:")
for path in required_paths:
    print(f"  {path.relative_to(PROJECT_DIR)}")
print(f"C sources: {len(c_sources)}")
print(f"Headers: {len(headers)}")

generated_files = sorted(
    path.relative_to(PROJECT_DIR)
    for path in PROJECT_DIR.rglob("*")
    if path.is_file()
)
print("Generated file sample:")
for relpath in generated_files[:25]:
    print(f"  {relpath}")
if len(generated_files) > 25:
    print(f"  ... {len(generated_files) - 25} more file(s)")

first_lines(PROJECT_DIR / f"{PROJECT_NAME}.par", n=8)

Required files:
  Makefile
  wave_equation_cartesian.par
C sources: 15
Headers: 6
Generated file sample:
  BHaH_defines.h
  BHaH_function_prototypes.h
  Makefile
  MoL/MoL_free_intermediate_stage_gfs.c
  MoL/MoL_malloc_intermediate_stage_gfs.c
  MoL/MoL_step_forward_in_time.c
  apply_bcs.c
  cmdline_input_and_parfile_parser.c
  commondata_struct_set_to_default.c
  diagnostics.c
  exact_solution_single_Cartesian_point.c
  griddata_free.c
  initial_data.c
  intrinsics/simd_intrinsics.h
  main.c
  numerical_grids_and_timestep.c
  params_struct_set_to_default.c
  progress_indicator.c
  rhs_eval.c
  set_CodeParameters-nopointer.h
  set_CodeParameters-simd.h
  set_CodeParameters.h
  wave_equation_cartesian.par
First 8 line(s) of wave_equation_cartesian.par:
#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_outpu

## Step 5: Build the Generated Project

This step requires `make` and a C compiler. If either is unavailable, project generation can still be valid, but full first-run validation cannot be completed.

In [5]:
require_tool("make")
require_c_compiler()

run_command(["make"], cwd=PROJECT_DIR)

executable = PROJECT_DIR / PROJECT_NAME
if not executable.is_file():
    raise FileNotFoundError(executable)
print(f"Executable: {executable.name}")

make: available
cc: available
gcc: available
clang: available
$ make
cwd: .


Last 16 output line(s):
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs.c -o apply_bcs.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c cmdline_input_and_parfile_parser.c -o cmdline_input_and_parfile_parser.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c commondata_struct_set_to_default.c -o commondata_struct_set_to_default.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c diagnostics.c -o diagnostics.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c exact_solution_single_Cartesian_point.c -o exact_solution_single_Cartesian_point.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c griddata_free.c -o griddata_free.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c initial_data.c -o initial_data.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c main.c -o main.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c MoL/MoL_free_intermediate_stage_gfs.c -o MoL/MoL_free_intermediate_stage_gfs.o
cc 

## Step 6: Run the Executable and Inspect Diagnostics

Run the default case, then rerun with `2.0` as the `convergence_factor` command-line override. The second run should write `out0d-conv_factor2.00.txt`.

In [6]:
executable = PROJECT_DIR / PROJECT_NAME
if not executable.is_file():
    raise FileNotFoundError(executable)

run_command([f"./{PROJECT_NAME}"], cwd=PROJECT_DIR)
run_command([f"./{PROJECT_NAME}", "2.0"], cwd=PROJECT_DIR)

diagnostic = PROJECT_DIR / "out0d-conv_factor2.00.txt"
if not diagnostic.is_file():
    raise FileNotFoundError(diagnostic)
first_lines(diagnostic, n=8)

$ ./wave_equation_cartesian
cwd: .
Last 40 output line(s):
It: 13 t=2.031 / 8.0 = 25.39% dt=1/6.4 | t/h=276980.54 ETA 0h00m00s
It: 14 t=2.188 / 8.0 = 27.34% dt=1/6.4 | t/h=279324.41 ETA 0h00m00s
It: 15 t=2.344 / 8.0 = 29.30% dt=1/6.4 | t/h=280473.06 ETA 0h00m00s
It: 16 t=2.500 / 8.0 = 31.25% dt=1/6.4 | t/h=282469.92 ETA 0h00m00s
It: 17 t=2.656 / 8.0 = 33.20% dt=1/6.4 | t/h=281551.53 ETA 0h00m00s
It: 18 t=2.812 / 8.0 = 35.16% dt=1/6.4 | t/h=282796.74 ETA 0h00m00s
It: 19 t=2.969 / 8.0 = 37.11% dt=1/6.4 | t/h=283584.68 ETA 0h00m00s
It: 20 t=3.125 / 8.0 = 39.06% dt=1/6.4 | t/h=284595.03 ETA 0h00m00s
It: 21 t=3.281 / 8.0 = 41.02% dt=1/6.4 | t/h=285938.80 ETA 0h00m00s
It: 22 t=3.438 / 8.0 = 42.97% dt=1/6.4 | t/h=279920.85 ETA 0h00m00s
It: 23 t=3.594 / 8.0 = 44.92% dt=1/6.4 | t/h=281087.90 ETA 0h00m00s
It: 24 t=3.750 / 8.0 = 46.88% dt=1/6.4 | t/h=282293.21 ETA 0h00m00s
It: 25 t=3.906 / 8.0 = 48.83% dt=1/6.4 | t/h=283485.38 ETA 0h00m00s
It: 26 t=4.062 / 8.0 = 50.78% dt=1/6.4 | t/h=284336.99 ET

Last 40 output line(s):
It: 64 t=5.000 / 8.0 = 62.50% dt=1/12.8 | t/h=7211.83 ETA 0h00m01s
It: 65 t=5.078 / 8.0 = 63.48% dt=1/12.8 | t/h=7207.24 ETA 0h00m01s
It: 66 t=5.156 / 8.0 = 64.45% dt=1/12.8 | t/h=7208.49 ETA 0h00m01s
It: 67 t=5.234 / 8.0 = 65.43% dt=1/12.8 | t/h=7209.55 ETA 0h00m01s
It: 68 t=5.312 / 8.0 = 66.41% dt=1/12.8 | t/h=7209.97 ETA 0h00m01s
It: 69 t=5.391 / 8.0 = 67.38% dt=1/12.8 | t/h=7211.33 ETA 0h00m01s
It: 70 t=5.469 / 8.0 = 68.36% dt=1/12.8 | t/h=7212.47 ETA 0h00m01s
It: 71 t=5.547 / 8.0 = 69.34% dt=1/12.8 | t/h=7204.18 ETA 0h00m01s
It: 72 t=5.625 / 8.0 = 70.31% dt=1/12.8 | t/h=7204.00 ETA 0h00m01s
It: 73 t=5.703 / 8.0 = 71.29% dt=1/12.8 | t/h=7205.59 ETA 0h00m01s
It: 74 t=5.781 / 8.0 = 72.27% dt=1/12.8 | t/h=7206.55 ETA 0h00m01s
It: 75 t=5.859 / 8.0 = 73.24% dt=1/12.8 | t/h=7207.44 ETA 0h00m01s
It: 76 t=5.938 / 8.0 = 74.22% dt=1/12.8 | t/h=7208.25 ETA 0h00m01s
It: 77 t=6.016 / 8.0 = 75.20% dt=1/12.8 | t/h=7208.36 ETA 0h00m00s
It: 78 t=6.094 / 8.0 = 76.17% dt=1/12.

## Step 7: Interpret the First Success Criterion

The first setup milestone is complete when generation succeeds, `make` succeeds, the executable runs, and diagnostic files appear. Deeper convergence analysis belongs in later wave-equation and validation notebooks. The `convergence_factor` value used above is a command-line override registered by the generator.

Rerun the generator rather than hand-editing generated C files when changing the generated project.

## Step 8: Where This Fits in the Tutorial

This setup unit leads into the package map and repository layout, generated-project anatomy, core tools (`params`, `grid`, `indexedexp`, `finite_difference`, `c_codegen`, `c_function`), wave-equation notebooks, and later reference-metric and BHaH workflows.

## Where This Leads

- [Index](../index.ipynb) reviews the prerequisite step.
- [NRPy Package Map](package_map.ipynb) continues the tutorial path.